# MAPPO Exploration To Forage 50x50

Continue the `50x50` exploration policy into regular foraging. Training keeps a small decaying new-cell bonus, but pickup and colony delivery rewards dominate the objective.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny job before loading the warm-start checkpoint.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Continuation Settings

Edit `experiments/exploration_to_forage_50x50.json` for durable reward-scale or budget changes.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_50x50.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "exploration_to_forage_50x50"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
if SOURCE_CHECKPOINT.name != "jax_mappo_explore_50x50.pkl":
    raise ValueError(f"Expected the 50x50 exploration checkpoint, got {SOURCE_CHECKPOINT}")
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Run the exploration curriculum first: {SOURCE_CHECKPOINT}")
experiment_args["load_model"] = str(SOURCE_CHECKPOINT)

GLOBAL_UPDATE_CAP = int(experiment.metadata["global_update_cap"])
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = "exploration_to_forage_50x50"
WANDB_MODE = "online"
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.SINGLE_CHECKPOINT_ARG_EXCLUDES,
)
COMMON_ARGS += ["--wandb-mode", WANDB_MODE, "--wandb-group", WANDB_GROUP]
if WANDB_PROJECT is not None:
    COMMON_ARGS += ["--wandb-project", WANDB_PROJECT]
if WANDB_ENTITY is not None:
    COMMON_ARGS += ["--wandb-entity", WANDB_ENTITY]
{
    "source_checkpoint": SOURCE_CHECKPOINT,
    "updates": GLOBAL_UPDATE_CAP,
    "reward_scales": experiment.metadata["reward_scales"],
}


## Train Continuation Checkpoint


In [ ]:
training_result = workflows.run_jax_checkpoint_training(
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    update_timesteps=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    progress_label="exploration-to-forage",
)
FINAL_CHECKPOINT_PATH = training_result["checkpoint_path"]
training_result


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_jax_checkpoint_rollout(
    run_dir=RUN_DIR,
    checkpoint_path=FINAL_CHECKPOINT_PATH,
    media_dir=MEDIA_DIR,
    rollout_filename="jax_mappo_exploration_to_forage_50x50_rollout.mp4",
    title="JAX MAPPO exploration-to-forage 50x50 rollout",
    description="Sampled rollout after continuing the 50x50 exploration policy into foraging.",
    metadata={"experiment_config": str(EXPERIMENT_CONFIG)},
    tile_size=ROLLOUT_TILE_SIZE,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=f"{WANDB_GROUP}_rollout",
    wandb_mode=WANDB_MODE,
    wandb_video_key="videos/exploration_to_forage/50x50",
    wandb_step=training_result["final_train_metrics"].get("global_step"),
)
rollout_result
